# 🎙 CS 5542 — Meeting Intelligence API Server

Optimized for **Google Colab GPU (T4 / A100)** — Whisper + pyannote diarization + Gemini summarization.

```
Browser (HTML app) → POST /analyze → ngrok → Flask + GPU → JSON results back
```

**Steps:**
1. Run Cell 1 — clone repo + install packages
2. Run Cell 2 — pre-load all heavy models (~2-3 min warm-up)
3. Run Cell 3 — Flask API routes
4. Run Cell 4 — start ngrok + Flask → copy URL into your HTML frontend

In [ ]:
# ── CELL 1 ── Clone GitHub repo + Install ───────────────────────────────────
import subprocess, sys, os
# ▼▼ EDIT: your GitHub repo URL ▼▼
GITHUB_REPO = 'https://github.com/mosomo82/COMP_SCI_5542.git'
# For a private repo use a token:
# GITHUB_REPO = 'https://YOUR_TOKEN@github.com/YOUR_USERNAME/COMP_SCI_5542'
# ▲▲─────────────────────────────▲▲
CLONE_DIR   = '/content/COMP_SCI_5542'
PROJECT_DIR = '/content/COMP_SCI_5542/AI Meeting Intelligent System Challenge'
if not os.path.exists(CLONE_DIR):
    subprocess.run(['git', 'clone', GITHUB_REPO, CLONE_DIR], check=True)
    print(f'Cloned to {CLONE_DIR}')
else:
    subprocess.run(['git', '-C', CLONE_DIR, 'pull'], check=True)
    print(f'Pulled latest changes into {CLONE_DIR}')
sys.path.insert(0, PROJECT_DIR)
os.chdir(PROJECT_DIR)
print(f'Working dir: {os.getcwd()}')
# Packages Colab pre-installs as a compatible set — upgrading any one of them
# independently causes breakage (torch/torchvision CUDA mismatch, numpy
# downgrade breaking scipy/_core/umath, etc.). Skip them here.
COLAB_MANAGED = ('torch', 'torchaudio', 'torchvision', 'numpy')
with open('requirements.txt') as f:
    reqs = [
        line.strip() for line in f
        if line.strip()
        and not line.startswith('#')
        and not any(line.lower().startswith(p) for p in COLAB_MANAGED)
    ]
subprocess.run([sys.executable, '-m', 'pip', 'install', '--quiet'] + reqs, check=True)
# Server extras (not in requirements.txt)
subprocess.run([sys.executable, '-m', 'pip', 'install', '--quiet',
                'flask', 'flask-cors', 'pyngrok'], check=True)
# Re-pin numpy to 2.x after all installs — transitive deps of pyannote/lightning
# can silently downgrade it to 1.x, breaking scipy and numpy._core.umath imports.
subprocess.run([sys.executable, '-m', 'pip', 'install', '--quiet', '-U', 'numpy>=2.0'], check=True)
print('Packages ready')
import torch, numpy as np
if torch.cuda.is_available():
    print(f'GPU  : {torch.cuda.get_device_name(0)}')
    print(f'VRAM : {torch.cuda.get_device_properties(0).total_memory / 1e9:.0f} GB')
    print(f'CUDA : {torch.version.cuda}')
print(f'torch: {torch.__version__}  numpy: {np.__version__}')

In [ ]:
# ── CELL 2 ── Pre-load all heavy models ─────────────────────────────────────
import os, sys, warnings, time
import torch
warnings.filterwarnings('ignore')
# Load secrets from Colab Secret Manager (recommended)
try:
    from google.colab import userdata
    for src, dst in [('HF_TOKEN_MEETING', 'HF_TOKEN'),
                     ('GEMINI_API_KEY_MEETING', 'GEMINI_API_KEY'),
                     ('ANTHROPIC_API_KEY', 'ANTHROPIC_API_KEY')]:
        val = userdata.get(src)
        if val:
            os.environ[dst] = val
            print(f'  {src} loaded from Colab Secrets')
except Exception:
    pass
# ▼▼ Or paste keys directly (less secure) ▼▼
# os.environ['HF_TOKEN']          = 'hf_...'
# os.environ['GEMINI_API_KEY']    = 'AIza...'
# os.environ['ANTHROPIC_API_KEY'] = 'sk-ant-...'
# ▲▲──────────────────────────────────────▲▲
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Device: {DEVICE}')
_whisper_model      = None
_diarize_pipeline   = None
_sentiment_pipeline = None
_keyword_model      = None
_tts_processor      = None
_tts_model          = None
_tts_vocoder        = None
_tts_speaker_embed  = None
def preload_models():
    global _whisper_model, _diarize_pipeline, _sentiment_pipeline
    global _keyword_model
    global _tts_processor, _tts_model, _tts_vocoder, _tts_speaker_embed
    # ── Whisper ──────────────────────────────────────────────────────────────
    t = time.time()
    print('Loading Whisper (small)...')
    import whisper
    _whisper_model = whisper.load_model('small', device=DEVICE)
    print(f'  Whisper ready ({time.time()-t:.1f}s)')
    # ── pyannote speaker diarization ─────────────────────────────────────────
    t = time.time()
    print('Loading pyannote speaker diarization...')
    hf_token = os.getenv('HF_TOKEN') # Using HF_TOKEN mapped from secrets
    if not hf_token:
        print('  WARNING: HF_TOKEN not set — diarization will be skipped')
    else:
        from pyannote.audio import Pipeline as PyannotePipeline
        from huggingface_hub import login
        login(token=hf_token)
        _diarize_pipeline = PyannotePipeline.from_pretrained(
            'pyannote/speaker-diarization-3.1'
        )
        if DEVICE == 'cuda':
            _diarize_pipeline = _diarize_pipeline.to(torch.device('cuda'))
        print(f'  pyannote ready ({time.time()-t:.1f}s)')
    # ── Sentiment ─────────────────────────────────────────────────────────────
    t = time.time()
    print('Loading sentiment classifier...')
    from transformers import pipeline as hf_pipeline
    _sentiment_pipeline = hf_pipeline(
        'sentiment-analysis',
        model='distilbert-base-uncased-finetuned-sst-2-english',
        device=0 if DEVICE == 'cuda' else -1,
    )
    print(f'  Sentiment ready ({time.time()-t:.1f}s)')
    # ── KeyBERT ───────────────────────────────────────────────────────────────
    t = time.time()
    print('Loading KeyBERT...')
    from keybert import KeyBERT
    _keyword_model = KeyBERT()
    print(f'  KeyBERT ready ({time.time()-t:.1f}s)')
    # ── SpeechT5 TTS ─────────────────────────────────────────────────────────
    t = time.time()
    print('Loading SpeechT5 TTS...')
    from transformers import SpeechT5Processor, SpeechT5ForTextToSpeech, SpeechT5HifiGan
    _tts_processor = SpeechT5Processor.from_pretrained('microsoft/speecht5_tts')
    _tts_model     = SpeechT5ForTextToSpeech.from_pretrained('microsoft/speecht5_tts').to(DEVICE)
    _tts_vocoder   = SpeechT5HifiGan.from_pretrained('microsoft/speecht5_hifigan').to(DEVICE)
    # Speaker embeddings — datasets 3.x dropped legacy dataset-script support,
    # so load from the auto-generated parquet export instead.
    import pandas as pd, requests, io as _io
    _hf = os.getenv('HF_TOKEN')
    _headers = {'Authorization': f'Bearer {_hf}'} if _hf else {}
    _url = (
        'https://huggingface.co/datasets/Matthijs/cmu-arctic-xvectors'
        '/resolve/refs%2Fconvert%2Fparquet/default/validation/0000.parquet'
    )
    _resp = requests.get(_url, headers=_headers)
    _resp.raise_for_status()
    _df = pd.read_parquet(_io.BytesIO(_resp.content))
    _tts_speaker_embed = torch.tensor(list(_df.iloc[7306]['xvector'])).unsqueeze(0).to(DEVICE)
    print(f'  SpeechT5 ready ({time.time()-t:.1f}s)')
    print('\nAll models preloaded!')
preload_models()


In [ ]:
# ── CELL 3 ── Flask API ──────────────────────────────────────────────────────
import base64, io, time, traceback, tempfile, json, os
from flask import Flask, request, jsonify
from flask_cors import CORS
import torch

app = Flask(__name__)
CORS(app)


def b64_to_audio_file(data_url: str) -> str:
    """Decode a base64 data URL to a temp .wav file, return its path."""
    header, data = data_url.split(',', 1)
    # Detect extension from data URL mime type
    ext = '.wav'
    if 'mp3' in header:
        ext = '.mp3'
    elif 'm4a' in header or 'mp4' in header:
        ext = '.m4a'
    tmp = tempfile.NamedTemporaryFile(suffix=ext, delete=False)
    tmp.write(base64.b64decode(data))
    tmp.close()
    return tmp.name


def audio_file_to_b64(path: str) -> str | None:
    """Encode an audio file to a base64 data URL."""
    if not path or not os.path.exists(path):
        return None
    with open(path, 'rb') as f:
        data = base64.b64encode(f.read()).decode()
    return f'data:audio/wav;base64,{data}'


def _run_pipeline_with_cached_models(audio_path, whisper_model, prompt_variant, generate_audio):
    """
    Run pipeline stages using pre-loaded models from Cell 2.
    This avoids reloading models on every request.
    """
    import numpy as np
    import soundfile as sf

    times = {}
    result = {}

    # ── Stage 1: Transcription ────────────────────────────────────────────────
    t0 = time.time()
    print('  Stage 1/5: Transcription')

    # Load a different Whisper size if requested
    if whisper_model != 'small':
        import whisper as whisper_mod
        wm = whisper_mod.load_model(whisper_model, device=DEVICE)
    else:
        wm = _whisper_model

    transcription = wm.transcribe(audio_path, fp16=(DEVICE == 'cuda'))
    result['transcript'] = transcription['text']
    result['segments']   = transcription.get('segments', [])
    times['transcription'] = round(time.time() - t0, 2)

    # ── Stage 2: Speaker Diarization ──────────────────────────────────────────
    t0 = time.time()
    print('  Stage 2/5: Speaker diarization')

    if _diarize_pipeline is not None:
        from src.diarize import diarize, format_diarized_transcript, save_diarized
        diarized = diarize(audio_path, result['segments'])
        result['diarized']      = diarized
        result['diarized_text'] = format_diarized_transcript(diarized)
    else:
        # Fallback: no diarization — use plain transcript
        result['diarized']      = []
        result['diarized_text'] = result['transcript']
        print('  WARNING: diarization skipped (HF_TOKEN_MEETING not set)')

    times['diarization'] = round(time.time() - t0, 2)

    # ── Stage 3: NLP Analysis ─────────────────────────────────────────────────
    t0 = time.time()
    print('  Stage 3/5: NLP analysis')
    from src.analyze import analyze_sentiment, extract_keywords
    sentiment = analyze_sentiment(result['diarized'])
    keywords  = extract_keywords(result['transcript'])
    result['sentiment'] = sentiment
    result['keywords']  = keywords
    times['analysis'] = round(time.time() - t0, 2)

    # ── Stage 4: LLM Summarization ────────────────────────────────────────────
    t0 = time.time()
    print('  Stage 4/5: LLM summarization')
    from src.summarize import summarize, format_summary_for_speech
    summary = summarize(
        diarized_transcript=result['diarized_text'],
        keywords=keywords,
        sentiment=sentiment,
        prompt_variant=prompt_variant,
    )
    result['summary']      = summary
    result['summary_text'] = format_summary_for_speech(summary)
    times['summarization'] = round(time.time() - t0, 2)

    # ── Stage 5: Text-to-Speech ───────────────────────────────────────────────
    if generate_audio and _tts_model is not None:
        t0 = time.time()
        print('  Stage 5/5: TTS narration')
        from src.speak import synthesize_speech
        audio_out = synthesize_speech(result['summary_text'])
        result['audio_path'] = audio_out
        times['tts'] = round(time.time() - t0, 2)
    else:
        result['audio_path'] = None
        print('  Stage 5/5: TTS skipped')

    result['stage_times'] = times
    return result


@app.route('/health', methods=['GET'])
def health():
    info = {
        'status': 'ok',
        'device': DEVICE,
        'models_loaded': {
            'whisper':   _whisper_model is not None,
            'diarize':   _diarize_pipeline is not None,
            'sentiment': _sentiment_pipeline is not None,
            'tts':       _tts_model is not None,
        },
    }
    if DEVICE == 'cuda':
        free, total = torch.cuda.mem_get_info()
        info['gpu']          = torch.cuda.get_device_name(0)
        info['vram_total_gb'] = round(total / 1e9, 1)
        info['vram_free_gb']  = round(free  / 1e9, 1)
    return jsonify(info)


@app.route('/analyze', methods=['POST'])
def analyze():
    """
    Analyze a meeting audio file.

    Request JSON:
    {
        "audio":          "data:audio/wav;base64,...",
        "whisper_model":  "small" | "medium" | "tiny"  (default: "small"),
        "prompt_variant": "baseline" | "improved"      (default: "improved"),
        "generate_audio": true | false                  (default: true)
    }

    Response JSON:
    {
        "transcript":    "...",
        "diarized_text": "SPEAKER_00: ... \nSPEAKER_01: ...",
        "summary":       { structured summary dict },
        "sentiment":     { per-speaker sentiment dict },
        "keywords":      [...],
        "summary_audio": "data:audio/wav;base64,..." | null,
        "stage_times":   { "transcription": 4.2, ... },
        "total_time_s":  45.2
    }
    """
    audio_path = None
    try:
        d = request.get_json(force=True)

        if 'audio' not in d:
            return jsonify({'error': 'Missing "audio" field in request body'}), 400

        # Decode audio from base64 to temp file
        audio_path = b64_to_audio_file(d['audio'])

        whisper_model  = d.get('whisper_model',  'small')
        prompt_variant = d.get('prompt_variant', 'improved')
        generate_audio = bool(d.get('generate_audio', True))

        print(f'\n[/analyze] whisper={whisper_model} prompt={prompt_variant} tts={generate_audio}')
        t_start = time.time()

        result = _run_pipeline_with_cached_models(
            audio_path, whisper_model, prompt_variant, generate_audio
        )

        total = round(time.time() - t_start, 1)
        print(f'[/analyze] done in {total}s')

        return jsonify({
            'transcript':    result.get('transcript', ''),
            'diarized_text': result.get('diarized_text', ''),
            'summary':       result.get('summary', {}),
            'sentiment':     result.get('sentiment', {}),
            'keywords':      result.get('keywords', []),
            'summary_audio': audio_file_to_b64(result.get('audio_path')),
            'stage_times':   result.get('stage_times', {}),
            'total_time_s':  total,
        })

    except Exception:
        tb = traceback.format_exc()
        print(tb)
        return jsonify({'error': tb}), 500

    finally:
        # Clean up temp audio file
        if audio_path and os.path.exists(audio_path):
            os.unlink(audio_path)


print('Flask API ready  |  GET /health  |  POST /analyze')

In [ ]:
# ── CELL 4 ── Start ngrok + Flask  (keep this cell running!) ─────────────────
from pyngrok import ngrok

PORT = 5000

# Ngrok requires an auth token. Get yours at: https://dashboard.ngrok.com/get-started/your-authtoken
try:
    from google.colab import userdata
    ngrok_token = userdata.get('NGROK_AUTH_TOKEN')
    if ngrok_token:
        ngrok.set_auth_token(ngrok_token)
        print('Loaded ngrok auth token from Colab Secrets.')
except Exception:
    pass

# If you prefer not to use Colab Secrets, uncomment the line below and paste your token:
# ngrok.set_auth_token('PASTE_YOUR_NGROK_TOKEN_HERE')

ngrok.kill()
tunnel = ngrok.connect(PORT, 'http')
url    = tunnel.public_url

print()
print('=' * 60)
print('  COLAB MEETING INTELLIGENCE API IS LIVE')
print('=' * 60)
print(f'  Public URL  : {url}')
print(f'  Health check: {url}/health')
print(f'  API endpoint: POST {url}/analyze')
print()
print('  Paste the Public URL into your HTML frontend')
print('  Runtime > Interrupt kernel to stop the server')
print('=' * 60)
print()

app.run(port=PORT, use_reloader=False, debug=False, threaded=True)
